# Train Custom Retriever (Colab Wrapper)

This notebook is a thin wrapper around `train_custom_retriever.py`.

It is intended for the current Drive-backed workflow:
- mount Google Drive
- enter the synced repo root
- install dependencies
- verify the generated custom dataset artifacts exist
- run the existing training script with a fresh Python process

Preferred data strategy:
- sync the existing generated `custom-dataset-loader/data/processed/` artifacts into the Drive-backed repo
- only rerun the custom dataset pipeline if those processed artifacts are unavailable


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
import os
from pathlib import Path

# Update this to your Drive-synced repo path before running.
REPO_ROOT = Path("/content/drive/MyDrive/UVM/Deep Learning/Final Project/repo-mirror/deep-learning-final-project")
if not REPO_ROOT.exists():
    raise FileNotFoundError(f"Repo root not found: {REPO_ROOT}")

os.chdir(REPO_ROOT)
print(f"Working directory: {Path.cwd()}")


Working directory: /content/drive/MyDrive/UVM/Deep Learning/Final Project/repo-mirror/deep-learning-final-project


In [3]:
%pip install -r requirements.txt
%pip install sentence-transformers transformers datasets accelerate peft trl scikit-learn matplotlib beautifulsoup4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 23.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [5]:
from pathlib import Path
import torch

required_paths = [
    Path("train_custom_retriever.py"),
    Path("custom_retriever_bridge.py"),
    Path("custom-dataset-loader/data/processed/pages_sanitized.jsonl"),
    Path("custom-dataset-loader/data/processed/links_table.jsonl"),
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required files for custom retriever training:\n- " + "\n- ".join(missing)
    )

for path in required_paths:
    print(f"Found: {path}")

print(f"CUDA available: {torch.cuda.is_available()}")


Found: train_custom_retriever.py
Found: custom_retriever_bridge.py
Found: custom-dataset-loader/data/processed/pages_sanitized.jsonl
Found: custom-dataset-loader/data/processed/links_table.jsonl
CUDA available: True


In [6]:
PAGES_PATH = "custom-dataset-loader/data/processed/pages_sanitized.jsonl"
LINKS_PATH = "custom-dataset-loader/data/processed/links_table.jsonl"
OUTPUT_PATH = "custom_all_embeddings"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MAX_PAGES = None
MAX_EDGES = 50000
EPOCHS = 5
BATCH_SIZE = 128
WARMUP_STEPS = 50
EVAL_FRACTION = 0.2
SEED = 42
DEVICE = "cuda"

print(f"Output checkpoint will be written to: {OUTPUT_PATH}")


Output checkpoint will be written to: custom_all_embeddings


In [7]:
import shlex

train_cmd = [
    "python",
    "train_custom_retriever.py",
    "--pages_path", PAGES_PATH,
    "--links_path", LINKS_PATH,
    "--output_path", OUTPUT_PATH,
    "--model_name", MODEL_NAME,
    "--max_edges", str(MAX_EDGES),
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--warmup_steps", str(WARMUP_STEPS),
    "--eval_fraction", str(EVAL_FRACTION),
    "--seed", str(SEED),
    "--device", DEVICE,
]

if MAX_PAGES is not None:
    train_cmd.extend(["--max_pages", str(MAX_PAGES)])

train_cmd_str = " ".join(shlex.quote(part) for part in train_cmd)
print(train_cmd_str)
!{train_cmd_str}


python train_custom_retriever.py --pages_path custom-dataset-loader/data/processed/pages_sanitized.jsonl --links_path custom-dataset-loader/data/processed/links_table.jsonl --output_path custom_all_embeddings --model_name sentence-transformers/all-MiniLM-L6-v2 --max_edges 50000 --epochs 5 --batch_size 128 --warmup_steps 50 --eval_fraction 0.2 --seed 42 --device cuda
/content/drive/MyDrive/UVM/Deep Learning/Final Project/repo-mirror/deep-learning-final-project/train_custom_retriever.py:10: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import InputExample, SentenceTransformer, losses
/content/drive/MyDrive/UVM/Deep Learning/Final Project/repo-mirror/deep-learning-final-project/train_custom_retriever.py:11: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a f

In [8]:
from pathlib import Path

output_path = Path(OUTPUT_PATH)
if not output_path.exists():
    raise FileNotFoundError(f"Expected checkpoint directory was not created: {output_path}")

expected_files = ["config_sentence_transformers.json", "modules.json"]
missing_output = [name for name in expected_files if not (output_path / name).exists()]

print(f"Checkpoint directory created: {output_path.resolve()}")
if missing_output:
    print(f"Warning: missing expected SentenceTransformer files: {missing_output}")
else:
    print("Checkpoint looks like a SentenceTransformer directory.")


Checkpoint directory created: /content/drive/MyDrive/UVM/Deep Learning/Final Project/repo-mirror/deep-learning-final-project/custom_all_embeddings
Checkpoint looks like a SentenceTransformer directory.
